# 04 · Testing ML Pipelines

**Goal:** write the three kinds of tests that keep an ML pipeline trustworthy, and see a test stop bad data.

| Test type | Question it answers | Runs |
|---|---|---|
| **Unit test** | Does this one function do what it should? | On every code change, in seconds |
| **Data validation** | Is today's data the shape and quality we expect? | Before training on new data |
| **Model quality test** | Is the trained model good enough to ship? | After training, before release |

Python's built-in `unittest` module is used: a test is a method whose name starts with `test_`, and `self.assert...`
calls decide pass or fail.

In [ ]:
%run ./Includes/_config

In [ ]:
import unittest
import numpy as np
import pandas as pd

## 1. The pipeline functions we want to test

In [ ]:
def clean(pdf: pd.DataFrame) -> pd.DataFrame:
    """Drop duplicate rows, cap impossible BMI values, fill missing numbers with the column median."""
    out = pdf.drop_duplicates().copy()
    out["BMI"] = out["BMI"].clip(lower=12, upper=80)
    return out.fillna(out.median(numeric_only=True))

def add_features(pdf: pd.DataFrame) -> pd.DataFrame:
    """Add an obesity flag and a count of risk factors."""
    out = pdf.copy()
    out["obese"] = (out["BMI"] >= 30).astype(int)
    out["risk_factors"] = out["HighBP"] + out["HighChol"] + out["obese"] + out["Smoker"]
    return out

def validate(sdf) -> list:
    """Return a list of data problems (empty list = data is fine)."""
    problems = []
    missing = set(FEATURES + [LABEL]) - set(sdf.columns)
    if missing:
        problems.append(f"missing columns: {sorted(missing)}")
    if sdf.count() < 10_000:
        problems.append("fewer than 10,000 rows")
    if sdf.filter(F.col(LABEL).isNull() | ~F.col(LABEL).isin(0, 1)).count() > 0:
        problems.append(f"{LABEL} must be 0 or 1")
    if sdf.filter((F.col("BMI") < 10) | (F.col("BMI") > 100)).count() > 0:
        problems.append("BMI outside 10–100")
    return problems

## 2. Write the tests

In [ ]:
class TestTransformations(unittest.TestCase):
    """Unit tests: tiny hand-made inputs where we know the right answer."""
    def setUp(self):
        self.pdf = pd.DataFrame({"BMI": [25.0, 35.0, 150.0, None], "HighBP": [1, 0, 1, 0],
                                 "HighChol": [0, 0, 1, 1], "Smoker": [1, 0, 0, 0]})

    def test_bmi_is_capped(self):
        self.assertLessEqual(clean(self.pdf)["BMI"].max(), 80)

    def test_no_missing_values_after_clean(self):
        self.assertFalse(clean(self.pdf).isna().any().any())

    def test_obese_flag(self):
        # the missing BMI is filled with the median (35), so the last row also counts as obese
        self.assertEqual(add_features(clean(self.pdf))["obese"].tolist(), [0, 1, 1, 1])

    def test_risk_factor_count(self):
        self.assertEqual(add_features(clean(self.pdf))["risk_factors"].tolist(), [2, 1, 3, 2])

class TestDataQuality(unittest.TestCase):
    """Data validation on the real table."""
    def test_table_passes_validation(self):
        self.assertEqual(validate(spark.table(TABLE)), [])

class TestModelQuality(unittest.TestCase):
    """The model must beat a minimum AUC before it may be released."""
    MIN_AUC = 0.75
    def test_model_beats_threshold(self):
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import roc_auc_score
        from sklearn.model_selection import train_test_split
        pdf = spark.table(TABLE).sample(fraction=0.3, seed=1).toPandas()
        X_tr, X_te, y_tr, y_te = train_test_split(pdf[FEATURES], pdf[LABEL], test_size=0.25, random_state=1)
        auc = roc_auc_score(y_te, LogisticRegression(max_iter=1000).fit(X_tr, y_tr).predict_proba(X_te)[:, 1])
        print(f"   model AUC = {auc:.4f} (minimum {self.MIN_AUC})")
        self.assertGreaterEqual(auc, self.MIN_AUC)

## 3. Run the test suite

In [ ]:
suite = unittest.TestSuite()
for case in (TestTransformations, TestDataQuality, TestModelQuality):
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(case))
result = unittest.TextTestRunner(verbosity=2).run(suite)
print(f"\nRan {result.testsRun} tests: {len(result.failures)} failures, {len(result.errors)} errors")
assert result.wasSuccessful(), "Some tests failed - see the report above"

## 4. See validation catch bad data
A copy of the data is corrupted on purpose: some BMI values become negative and some labels become 2.
In a real pipeline this check runs **before** training, so a broken data feed never produces a model.

In [ ]:
bad = (spark.table(TABLE)
       .withColumn("BMI", F.when(F.col("id") % 50 == 0, -5.0).otherwise(F.col("BMI")))
       .withColumn(LABEL, F.when(F.col("id") % 97 == 0, 2.0).otherwise(F.col(LABEL))))

problems = validate(bad)
if problems:
    print("VALIDATION FAILED - training would be stopped:")
    for p in problems:
        print("  -", p)
else:
    print("Data is valid")

## Key points
- **Unit tests** use tiny inputs with known answers, so they run in seconds on every change.
- **Data validation** guards the pipeline's input; it is the cheapest place to catch problems.
- **Model quality tests** act as a release gate. Notebook 05 puts this gate inside an automated job.